In [0]:
%sql
CREATE EXTERNAL LOCATION raw_location
URL 'abfss://raw@stbankinglake21.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL banking_credential);

CREATE EXTERNAL LOCATION bronze_location
URL 'abfss://bronze@stbankinglake21.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL banking_credential);

CREATE EXTERNAL LOCATION silver_location
URL 'abfss://silver@stbankinglake21.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL banking_credential);

CREATE EXTERNAL LOCATION gold_location
URL 'abfss://gold@stbankinglake21.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL banking_credential);

In [0]:
%sql
-- Create Catalog
CREATE CATALOG banking_catalog
MANAGED LOCATION 'abfss://bronze@stbankinglake21.dfs.core.windows.net/';

-- Create Schemas for each layer
CREATE SCHEMA banking_catalog.bronze
MANAGED LOCATION 'abfss://bronze@stbankinglake21.dfs.core.windows.net/';

CREATE SCHEMA banking_catalog.silver
MANAGED LOCATION 'abfss://silver@stbankinglake21.dfs.core.windows.net/';

CREATE SCHEMA banking_catalog.gold
MANAGED LOCATION 'abfss://gold@stbankinglake21.dfs.core.windows.net/';

In [0]:

# Databricks Notebook: 01_Bronze_Ingestion

# Read all CSV files from raw container
raw_path = "abfss://raw@stbankinglake21.dfs.core.windows.net/"

accounts_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}accounts.csv")
atm_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}atm_transactions.csv")
branches_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}branches.csv")
credit_cards_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}credit_cards.csv")
customers_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}customers.csv")
employees_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}employees.csv")
fraud_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}fraud_alerts.csv")
kyc_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}kyc_documents.csv")
loans_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}loans.csv")
transactions_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}transactions.csv")

print("✅All CSV files loaded successfully")

✅All CSV files loaded successfully


In [0]:

%python
# Write each DataFrame as Delta table in Bronze layer
bronze_path = "abfss://bronze@stbankinglake21.dfs.core.windows.net/"

# Accounts
accounts_df.write.format("delta").mode("overwrite").save(f"{bronze_path}accounts")

# ATM Transactions
atm_df.write.format("delta").mode("overwrite").save(f"{bronze_path}atm_transactions")

# Branches
branches_df.write.format("delta").mode("overwrite").save(f"{bronze_path}branches")

# Credit Cards
credit_cards_df.write.format("delta").mode("overwrite").save(f"{bronze_path}credit_cards")

# Customers
customers_df.write.format("delta").mode("overwrite").save(f"{bronze_path}customers")

# Employees
employees_df.write.format("delta").mode("overwrite").save(f"{bronze_path}employees")

# Fraud Alerts
fraud_df.write.format("delta").mode("overwrite").save(f"{bronze_path}fraud_alerts")

# KYC Documents
kyc_df.write.format("delta").mode("overwrite").save(f"{bronze_path}kyc_documents")

# Loans
loans_df.write.format("delta").mode("overwrite").save(f"{bronze_path}loans")

# Transactions
transactions_df.write.format("delta").mode("overwrite").save(f"{bronze_path}transactions")

print("✅ All tables written to Bronze layer!")

✅ All tables written to Bronze layer!


In [0]:

%sql
-- Create external tables pointing to Bronze Delta files
CREATE TABLE banking_catalog.bronze.accounts
USING DELTA
LOCATION 'abfss://bronze@stbankinglake21.dfs.core.windows.net/accounts/';

CREATE TABLE banking_catalog.bronze.atm_transactions
USING DELTA
LOCATION 'abfss://bronze@stbankinglake21.dfs.core.windows.net/atm_transactions/';

CREATE TABLE banking_catalog.bronze.branches
USING DELTA
LOCATION 'abfss://bronze@stbankinglake21.dfs.core.windows.net/branches/';

CREATE TABLE banking_catalog.bronze.credit_cards
USING DELTA
LOCATION 'abfss://bronze@stbankinglake21.dfs.core.windows.net/credit_cards/';

CREATE TABLE banking_catalog.bronze.customers
USING DELTA
LOCATION 'abfss://bronze@stbankinglake21.dfs.core.windows.net/customers/';

CREATE TABLE banking_catalog.bronze.employees
USING DELTA
LOCATION 'abfss://bronze@stbankinglake21.dfs.core.windows.net/employees/';

CREATE TABLE banking_catalog.bronze.fraud_alerts
USING DELTA
LOCATION 'abfss://bronze@stbankinglake21.dfs.core.windows.net/fraud_alerts/';

CREATE TABLE banking_catalog.bronze.kyc_documents
USING DELTA
LOCATION 'abfss://bronze@stbankinglake21.dfs.core.windows.net/kyc_documents/';

CREATE TABLE banking_catalog.bronze.loans
USING DELTA
LOCATION 'abfss://bronze@stbankinglake21.dfs.core.windows.net/loans/';

CREATE TABLE banking_catalog.bronze.transactions
USING DELTA
LOCATION 'abfss://bronze@stbankinglake21.dfs.core.windows.net/transactions/';
